# Protocol A training
Use the MSLD v2.0 release, the extracted Protocol A frozen input directory and the release code dataset. Run the preflight before the GPU job.


In [ ]:
# Cell 1 - install the small required packages; preserve Kaggle's GPU PyTorch.
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys
required = {'scikit-learn': '1.6.1', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'python-docx': '1.2.0', 'reportlab': '4.4.9', 'timm': '1.0.29'}
changes = []
for name, version in required.items():
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        changes.append(f'{name}=={version}')
if changes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *changes], check=True)
for module, package in {'lxml': 'lxml', 'contourpy': 'contourpy', 'cycler': 'cycler', 'fontTools': 'fonttools', 'kiwisolver': 'kiwisolver', 'pyparsing': 'pyparsing', 'dateutil': 'python-dateutil', 'charset_normalizer': 'charset-normalizer', 'huggingface_hub': 'huggingface_hub', 'safetensors': 'safetensors'}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', package], check=True)
print('Setup complete. Continue to Cell 2.')


In [ ]:
from pathlib import Path
import json
import os
CODE_DIR = Path('/kaggle/working/msld_final_code')
from pathlib import Path
import subprocess
import sys

code_tools = sorted(Path('/kaggle/input').rglob('materialize_workflow.py'))
code_tools = [p for p in code_tools if (p.parents[1] / 'reproducibility/code_sources.csv').is_file()]
if len(code_tools) != 1:
    raise FileNotFoundError('Attach one extracted release code dataset. See docs/reproduce.md.')
subprocess.run([sys.executable, str(code_tools[0]), 'protocol_a', str(CODE_DIR)], check=True)
EXPECTED_A_SHA = 'b880264f09050c3dd6254c45eb4dfc579eb5cb55048a370f7f5f34271e639407'


In [ ]:
# Cell 3 - verify the frozen design and SHA-256 contents of all 755 images.
base_command = [sys.executable, str(CODE_DIR / 'run_protocol_a.py'),
                '--expected-manifest-sha', EXPECTED_A_SHA]
subprocess.run([*base_command, '--check-only'], check=True)


In [ ]:
# Cell 4 - run all 20 model/fold jobs, or resume the active job automatically.
environment = os.environ.copy()
environment['PYTHONHASHSEED'] = '42'
environment['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
environment['MPLCONFIGDIR'] = '/kaggle/working/.publication_plot_cache'
subprocess.run([*base_command, '--max-hours', '10.5'], env=environment, check=True)


In [ ]:
# Cell 5 - check completion and locate the final download.
from IPython.display import display, FileLink
status = json.loads(Path('/kaggle/working/protocol_a_755/run_status.json').read_text())
print(json.dumps(status, indent=2))
if status['status'] == 'complete':
    archive = Path('/kaggle/working/Protocol_A_755_Results_and_Publication.zip')
    assert archive.is_file()
    display(FileLink(str(archive)))
    print('Keep the FULL saved notebook outputs separately for all selected model weights.')
else:
    print('Progress was saved. Attach this FULL saved notebook output to this SAME notebook, keep the original two inputs, and Save & Run All again.')
    print('Completed jobs are skipped. The active job resumes from the last intact completed-epoch checkpoint.')


## Saving and resuming

For a safe pause, let the saved version finish. Attach its **full saved output** to this same notebook in addition to the original input datasets. Keep the same Kaggle environment and notebook code. Save & Run All again. An incomplete epoch is repeated from the preceding saved epoch.

Finished jobs retain both selected `.pt` weights and prediction records; their large optimizer checkpoints are removed after verification to keep the complete run within storage limits. The active job has alternating full checkpoints with optimizer, scheduler, scaler and random states. The final small ZIP excludes weights; the full saved Kaggle version retains them.

Do not use partial runs as final results. The success marker requires all 20 jobs, all 25 epochs per job, ensemble analysis and publication outputs.
